In [1]:
import os
import re
import pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from scipy.stats import norm
from sklearn.metrics import classification_report

DATA_DIR  = "/kaggle/input/datasets/rjxxxxx/bayesian"
MODEL_DIR = "/kaggle/working"

# ── 1. Load data ──
iu = pd.read_parquet(os.path.join(DATA_DIR, "instance_usage_training01_parquet.parquet"))
print("iu shape:", iu.shape)

# ── 2. Feature engineering ──
EPS = 1e-7

def parse_dist(val, expected_len=11):
    try:
        if isinstance(val, (list, np.ndarray)):
            vals = list(val)
        elif isinstance(val, str):
            nums = re.findall(r'[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?', val)
            vals = [float(x) for x in nums]
        else:
            return [np.nan] * expected_len
        return vals[:expected_len] + [np.nan] * max(0, expected_len - len(vals))
    except:
        return [np.nan] * expected_len

def get_stats(arr):
    arr   = np.array(arr, dtype=float)
    clean = arr[~np.isnan(arr)]
    if len(clean) == 0:
        stats = [np.nan] * 4
    else:
        stats = [np.mean(clean), np.max(clean),
                 np.percentile(clean, 90), np.mean(clean > 0)]
    buckets = list(arr[:9]) + [np.nan] * max(0, 9 - len(arr))
    return stats + buckets

pct_cols    = [f"cpu_p{i}" for i in range(0, 101, 10)]
dist_parsed = iu["cpu_usage_distribution"].apply(parse_dist)
dist_df     = pd.DataFrame(dist_parsed.tolist(), columns=pct_cols, index=iu.index)
iu          = pd.concat([iu, dist_df], axis=1)

iu["cpu_burstiness"] = iu["cpu_p90"] - iu["cpu_p10"]
iu["tail_cpu_mean"]  = iu[["cpu_p50", "cpu_p90", "cpu_p100"]].mean(axis=1)

tail_parsed = iu["tail_cpu_usage_distribution"].apply(
    lambda x: get_stats(parse_dist(x, expected_len=9))
)
stat_cols   = ['tail_cpu_mean', 'tail_cpu_max', 'tail_cpu_p90', 'tail_cpu_nonzero_frac']
bucket_cols = [f'cpu_p{i}' for i in range(91, 100)]
tail_df     = pd.DataFrame(tail_parsed.tolist(),
                           columns=stat_cols + bucket_cols, index=iu.index)
for col in bucket_cols:
    iu[col] = tail_df[col]

print("Feature engineering done.")

iu shape: (3436845, 15)
Feature engineering done.


In [2]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report

class StateDataset(Dataset):
    def __init__(self, seqs):
        self.seqs = seqs
    def __len__(self):
        return len(self.seqs)
    def __getitem__(self, idx):
        x, y = self.seqs[idx]
        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.long)

class ResourceTransformer(nn.Module):
    def __init__(self, input_dim=5, d_model=64, nhead=4, num_layers=2, dropout=0.1):
        super().__init__()
        self.input_proj  = nn.Linear(input_dim, d_model)
        encoder_layer    = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=128, dropout=dropout,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.classifier  = nn.Sequential(
            nn.Linear(d_model, 32),
            nn.ReLU(),
            nn.Linear(32, 3)
        )

    def forward(self, x):
        x = self.input_proj(x)
        x = self.transformer(x)
        x = x.mean(dim=1)
        return self.classifier(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [3]:
with open("/kaggle/input/datasets/rjxxxxx/bayesian/hmm_model_march8.pkl", "rb") as f:
    data = pickle.load(f)

mu          = np.array(data["mu"])
sigma       = np.array(data["sigma"])
norm_params = data["norm_params"]
features    = data["features"]

obs_list = []
for col_normed in features:
    col_raw  = col_normed.replace("_logged_normed", "")
    mean     = norm_params[col_raw]["mean"]
    std      = norm_params[col_raw]["std"]
    log_vals = np.log(iu[col_raw].astype(np.float32) + 1e-7)
    normed   = (log_vals - mean) / (std + 1e-8)
    obs_list.append(normed.values)

obs = np.nan_to_num(np.stack(obs_list, axis=1), nan=0.0)

log_probs = []
for k in range(mu.shape[0]):
    lp = norm.logpdf(obs, mu[k], sigma[k]).sum(axis=-1)
    log_probs.append(lp)

iu["state"] = np.argmax(np.stack(log_probs), axis=0)

print("State distribution:")
for k in range(3):
    print(f"  State {k}: {(iu['state']==k).mean():.2%}")

State distribution:
  State 0: 19.46%
  State 1: 44.72%
  State 2: 35.82%


In [4]:
# ── Build sequences (task-wise split) ──
SEQ_LEN  = 50
MIN_LEN  = 10

iu_sorted = iu.sort_values(["collection_id", "instance_index", "start_time"])

all_tasks   = list(iu_sorted.groupby(["collection_id", "instance_index"]).groups.keys())
split_idx   = int(0.8 * len(all_tasks))
train_tasks = set(all_tasks[:split_idx])
test_tasks  = set(all_tasks[split_idx:])

print(f"Train tasks: {len(train_tasks)}, Test tasks: {len(test_tasks)}")

train_seqs, test_seqs = [], []
for (cid, iidx), grp in iu_sorted.groupby(["collection_id", "instance_index"], sort=False):
    grp = grp.sort_values("start_time")
    if len(grp) < MIN_LEN:
        continue
    
    obs_list = []
    for col_normed in data["features"]:
        col_raw  = col_normed.replace("_logged_normed", "")
        mean     = norm_params[col_raw]["mean"]
        std      = norm_params[col_raw]["std"]
        log_vals = np.log(grp[col_raw].astype(np.float32) + 1e-7)
        normed   = (log_vals - mean) / (std + 1e-8)
        obs_list.append(normed.values)
    
    obs_5  = np.nan_to_num(np.stack(obs_list, axis=1), nan=0.0)
    states = grp["state"].values.astype(np.int64)
    seqs   = [(obs_5[t:t+SEQ_LEN], states[t+SEQ_LEN])
              for t in range(len(obs_5) - SEQ_LEN)]
    if (cid, iidx) in train_tasks:
        train_seqs.extend(seqs)
    else:
        test_seqs.extend(seqs)

print(f"Train: {len(train_seqs)}, Test: {len(test_seqs)}")

import random
random.seed(42)
train_sample = random.sample(train_seqs, len(train_seqs)//5)
test_sample  = random.sample(test_seqs,  len(test_seqs)//5)
print(f"Sampled Train: {len(train_sample)}, Test: {len(test_sample)}")

train_dl = DataLoader(StateDataset(train_sample), batch_size=512, shuffle=True)
test_dl  = DataLoader(StateDataset(test_sample),  batch_size=512, shuffle=False)

Train tasks: 724, Test tasks: 181
Train: 3306342, Test: 96017
Sampled Train: 661268, Test: 19203


In [10]:
model     = ResourceTransformer(input_dim=11).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

N_EPOCHS = 30
for epoch in range(N_EPOCHS):
    model.train()
    total_loss = 0
    for x, y in train_dl:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:3d} | Loss: {total_loss/len(train_dl):.4f}")

Epoch   5 | Loss: 0.5910
Epoch  10 | Loss: 0.5873
Epoch  15 | Loss: 0.5850
Epoch  20 | Loss: 0.5838
Epoch  25 | Loss: 0.5826
Epoch  30 | Loss: 0.5816


In [11]:
# ── Evaluate ──
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for x, y in test_dl:
        preds = model(x.to(device)).argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(y.numpy())

print("\nClassification Report:")
print(classification_report(all_labels, all_preds,
      target_names=["State 0", "State 1", "State 2"]))


Classification Report:
              precision    recall  f1-score   support

     State 0       0.61      0.09      0.15      1821
     State 1       0.61      0.99      0.75     11110
     State 2       0.83      0.12      0.21      6272

    accuracy                           0.62     19203
   macro avg       0.68      0.40      0.37     19203
weighted avg       0.68      0.62      0.52     19203



In [15]:
import torch
import json

torch.save(model.state_dict(), "/kaggle/working/transformer_final.pt")

results = {
    "accuracy": 0.62,
    "state_distribution": {"State 0": 0.1946, "State 1": 0.4472, "State 2": 0.3582},
    "classification_report": {
        "State 0": {"precision": 0.61, "recall": 0.09, "f1": 0.15},
        "State 1": {"precision": 0.61, "recall": 0.99, "f1": 0.75},
        "State 2": {"precision": 0.83, "recall": 0.12, "f1": 0.21},
        "macro_avg": {"precision": 0.68, "recall": 0.40, "f1": 0.37},
        "weighted_avg": {"precision": 0.68, "recall": 0.62, "f1": 0.52}
    },
    "model_config": {
        "input_dim": 11,
        "d_model": 64,
        "nhead": 4,
        "num_layers": 2,
        "seq_len": 50,
        "epochs": 15,
        "batch_size": 512,
        "lr": 1e-3,
        "hmm_source": "hmm_model_march8.pkl",
        "split": "task-wise 80/20",
        "subsample": "20%"
    }
}

with open("/kaggle/working/transformer_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("Saved.")

Saved.
